# Análisis exploratorio: anuncios de vehículos en EE. UU.

Dataset: `vehicles_us.csv` (anuncios publicados entre mayo de 2018 y abril de 2019).

**Secciones**
1. Carga y vista general
2. Valores ausentes y duplicados
3. Preparación de columnas
4. Distribuciones numéricas
5. Variables categóricas
6. Relaciones con el precio
7. Tiempo de publicación
8. Hallazgos

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

# El notebook está en la carpeta notebooks/, el CSV en la carpeta superior
car_data = pd.read_csv("/Users/Sebastian/VScode/Proyecto_sprint7/sprint7_proyecto/vehicles_us.csv")

## 1. Carga y vista general

In [3]:
print(car_data.shape)
car_data.info()

(51525, 13)
<class 'pandas.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  str    
 3   condition     51525 non-null  str    
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  str    
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  str    
 8   type          51525 non-null  str    
 9   paint_color   42258 non-null  str    
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  str    
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), str(7)
memory usage: 7.7 MB


In [4]:
car_data.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


In [5]:
car_data.describe().T

,count,mean,std,min,25%,50%,75%,max
price,51525.0,12132.464920,10040.803015,1.0,5000.0,9000.0,16839.0,375000.0
model_year,47906.0,2009.750470,6.282065,1908.0,2006.0,2011.0,2014.0,2019.0
cylinders,46265.0,6.125235,1.660360,3.0,4.0,6.0,8.0,12.0
odometer,43633.0,115553.461738,65094.611341,0.0,70000.0,113000.0,155000.0,990000.0
is_4wd,25572.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0
days_listed,51525.0,39.554760,28.204270,0.0,19.0,33.0,53.0,271.0


## 2. Valores ausentes y duplicados

In [6]:
missing = car_data.isna().sum().rename("faltantes").to_frame()
missing["porcentaje"] = (missing["faltantes"] / len(car_data) * 100).round(1)
missing = missing[missing["faltantes"] > 0].sort_values("faltantes", ascending=False)
missing = missing.rename_axis("columna").reset_index()
display(missing)

fig = px.bar(
    missing, x="columna", y="porcentaje", text="porcentaje",
    title="Porcentaje de valores ausentes por columna",
    labels={"porcentaje": "% ausente", "columna": "Columna"},
)
fig.show()

,columna,faltantes,porcentaje
0,is_4wd,25953,50.4
1,paint_color,9267,18.0
2,odometer,7892,15.3
3,cylinders,5260,10.2
4,model_year,3619,7.0


In [7]:
print("Filas duplicadas:", car_data.duplicated().sum())
print("Valores únicos de is_4wd:", car_data["is_4wd"].unique())

Filas duplicadas: 0
Valores únicos de is_4wd: [ 1. nan]


`is_4wd` solo contiene `1.0` o vacío. **Suposición:** los vacíos significan "no es 4x4", así que se rellenan con 0. El resto de columnas con ausentes (`paint_color`, `odometer`, `cylinders`, `model_year`) se dejan como `NaN` para no inventar datos.

## 3. Preparación de columnas

In [8]:
df = car_data.copy()
df["date_posted"] = pd.to_datetime(df["date_posted"])
df["is_4wd"] = df["is_4wd"].fillna(0).astype(int)
df["marca"] = df["model"].str.split().str[0]
df["antiguedad"] = (df["date_posted"].dt.year - df["model_year"]).clip(lower=0)

# Límites para ver mejor las gráficas (los valores extremos distorsionan la escala)
precio_p99 = df["price"].quantile(0.99)
odometro_p99 = df["odometer"].quantile(0.99)
print("Percentil 99 del precio:", precio_p99)
print("Percentil 99 del odómetro:", odometro_p99)
print("Anuncios con precio menor a 100:", (df["price"] < 100).sum())
print("Anuncios con precio menor a 1000:", (df["price"] < 1000).sum())

Percentil 99 del precio: 43995.0
Percentil 99 del odómetro: 286800.0000000002
Anuncios con precio menor a 100: 863
Anuncios con precio menor a 1000: 1404


## 4. Distribuciones numéricas

In [9]:
fig = px.histogram(df, x="price", nbins=80, range_x=[0, precio_p99],
                   title="Distribución del precio (hasta el percentil 99)")
fig.show()

In [10]:
fig = px.histogram(df, x="odometer", nbins=80, range_x=[0, odometro_p99],
                   title="Distribución del odómetro (hasta el percentil 99)")
fig.show()

In [11]:
fig = px.histogram(df[df["model_year"] >= 1990], x="model_year", nbins=30,
                   title="Distribución del año del modelo (desde 1990)")
fig.show()

In [12]:
fig = px.histogram(df, x="days_listed", nbins=60,
                   title="Días que permanece publicado un anuncio")
fig.show()

## 5. Variables categóricas

In [13]:
for col in ["type", "condition", "fuel", "transmission", "paint_color"]:
    conteo = (df[col].fillna("sin dato").value_counts()
              .rename_axis(col).reset_index(name="anuncios"))
    fig = px.bar(conteo, x=col, y="anuncios", title=f"Anuncios por {col}")
    fig.show()

In [14]:
top_modelos = (df["model"].value_counts().head(15)
               .rename_axis("modelo").reset_index(name="anuncios"))
fig = px.bar(top_modelos.sort_values("anuncios"), x="anuncios", y="modelo",
             orientation="h", title="15 modelos con más anuncios")
fig.show()

## 6. Relaciones con el precio

In [15]:
mask = (df["price"] <= precio_p99) & (df["odometer"] <= odometro_p99)
muestra = df[mask].sample(5000, random_state=42)

fig = px.scatter(muestra, x="odometer", y="price", color="condition", opacity=0.5,
                 title="Precio vs. odómetro (muestra de 5,000 anuncios)")
fig.show()

In [16]:
orden_condicion = ["salvage", "fair", "good", "excellent", "like new", "new"]
fig = px.box(df[df["price"] <= precio_p99], x="condition", y="price",
             category_orders={"condition": orden_condicion},
             title="Precio según la condición")
fig.show()

In [17]:
fig = px.box(df[df["price"] <= precio_p99], x="type", y="price",
             title="Precio según el tipo de vehículo")
fig.show()

In [18]:
precio_por_anio = (df[df["model_year"] >= 1990]
                   .groupby("model_year")["price"].median().reset_index())
fig = px.line(precio_por_anio, x="model_year", y="price", markers=True,
              title="Precio mediano por año del modelo")
fig.show()

In [19]:
fig = px.box(df[df["price"] <= precio_p99], x="is_4wd", y="price",
             title="Precio según tracción 4x4 (1 = sí)")
fig.show()

In [20]:
corr = df[["price", "odometer", "antiguedad", "cylinders", "days_listed"]].corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Matriz de correlación")
fig.show()

## 7. Tiempo de publicación

In [21]:
por_mes = (df.set_index("date_posted").resample("MS").size()
           .rename("anuncios").reset_index())
fig = px.bar(por_mes, x="date_posted", y="anuncios",
             title="Anuncios publicados por mes")
fig.show()

In [22]:
dias_por_tipo = (df.groupby("type")["days_listed"].median()
                 .sort_values().rename("mediana_dias").reset_index())
fig = px.bar(dias_por_tipo, x="type", y="mediana_dias",
             title="Mediana de días publicado según el tipo de vehículo")
fig.show()

## 8. Hallazgos

- **Tamaño:** 51,525 anuncios y 13 columnas, sin filas duplicadas.
- **Ausentes:** `is_4wd` (50 %), `paint_color` (18 %), `odometer` (15 %), `cylinders` (10 %) y `model_year` (7 %).
- **Precio:** mediana de 9,000 y media de 12,132, con una cola larga hacia precios altos (máximo 375,000). Hay 1,404 anuncios por debajo de 1,000, muchos con precio de 1, que probablemente son valores placeholder.
- **Odómetro:** mediana cercana a 113,000 con valores extremos (máximo 990,000).
- **Qué influye en el precio:** baja con el kilometraje (correlación ≈ -0.42) y con la antigüedad (≈ -0.43), y sube con el número de cilindros (≈ 0.30). Por condición, la mediana va de 2,500 (`fair`, `salvage`) a 21,999 (`new`).
- **Tipo de vehículo:** `truck` y `pickup` son los más caros (medianas de ~15,000 y ~14,200); `hatchback` y `sedan` los más baratos (~6,000).
- **Combustible:** 92 % gasolina.
- **Modelos más frecuentes:** ford f-150, chevrolet silverado 1500 y ram 1500.
- **Días publicado:** la correlación con las demás variables es prácticamente cero; el precio no explica cuánto tarda en venderse un anuncio.
- **Periodo:** del 1 de mayo de 2018 al 19 de abril de 2019.